In [1]:
import os
import time
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("All imports successful!")

torch: 2.4.0+cu121
CUDA: True
All imports successful!


In [2]:
# Our document corpus - 10 MLOps/AI documents
documents = [
    {
        "id": "doc1",
        "title": "RAG Architecture",
        "content": """Retrieval-Augmented Generation (RAG) is an AI architecture that enhances large language model outputs by retrieving relevant information from external knowledge bases. RAG consists of two main phases: indexing and querying. During indexing, documents are chunked, embedded into vectors, and stored in a vector database. During querying, the user query is embedded and used to find similar chunks via similarity search. The retrieved chunks are then passed as context to the LLM for grounded response generation."""
    },
    {
        "id": "doc2", 
        "title": "Vector Databases",
        "content": """Vector databases store high-dimensional embeddings and enable efficient similarity search. Popular vector databases include FAISS, Chroma, Weaviate, and Pinecone. FAISS (Facebook AI Similarity Search) is an open-source library optimized for fast nearest neighbor search. It supports multiple index types including Flat, IVF, and HNSW. Flat indexes provide exact search while IVF and HNSW provide approximate nearest neighbor search for better scalability."""
    },
    {
        "id": "doc3",
        "title": "Document Chunking",
        "content": """Document chunking is the process of splitting large documents into smaller pieces for embedding and retrieval. Common chunking strategies include fixed-size chunking, recursive character splitting, and semantic chunking. Fixed-size chunking splits documents into equal sized chunks. Recursive splitting tries to split on natural boundaries like paragraphs and sentences. Chunk size and overlap are critical parameters - typical chunk sizes range from 256 to 1024 tokens with 10-20% overlap to preserve context across chunk boundaries."""
    },
    {
        "id": "doc4",
        "title": "Embedding Models",
        "content": """Embedding models convert text into dense numerical vectors that capture semantic meaning. Popular embedding models include all-MiniLM-L6-v2, all-mpnet-base-v2, and bge-large-en-v1.5. The all-MiniLM-L6-v2 model produces 384-dimensional embeddings and is optimized for speed. Semantic similarity between texts is measured using cosine similarity or dot product between their embedding vectors. Models trained with contrastive learning produce embeddings where semantically similar texts are close together in vector space."""
    },
    {
        "id": "doc5",
        "title": "LLM Hallucination",
        "content": """Hallucination in large language models refers to the generation of factually incorrect or fabricated information. RAG systems help reduce hallucinations by grounding responses in retrieved context. Hallucinations occur when models rely on parametric knowledge rather than provided context. Evaluation of hallucinations involves checking if response claims are supported by retrieved documents. Common mitigation strategies include strict prompting to only use provided context, citation requirements, and faithfulness scoring."""
    },
    {
        "id": "doc6",
        "title": "Agentic AI Systems",
        "content": """Agentic AI systems are LLM-powered applications that can reason, plan, and execute multi-step tasks autonomously. Key components include planning modules, tool use capabilities, memory management, and self-correction mechanisms. The ReAct pattern combines reasoning and acting by alternating between thought steps and action steps. Agents can use tools like web search, code execution, and retrieval systems. Multi-agent systems coordinate multiple specialized agents to solve complex tasks."""
    },
    {
        "id": "doc7",
        "title": "MLOps Pipeline",
        "content": """MLOps refers to the practice of applying DevOps principles to machine learning systems. Key components of MLOps include data versioning, model training pipelines, model registry, serving infrastructure, and monitoring. Continuous integration and deployment for ML models involves automated testing of data quality, model performance, and serving endpoints. Model drift detection monitors for changes in input data distribution or model output quality over time."""
    },
    {
        "id": "doc8",
        "title": "Retrieval Evaluation",
        "content": """Retrieval evaluation measures the quality of document retrieval in RAG systems. Key metrics include Precision@K which measures the fraction of retrieved documents that are relevant, Recall@K which measures the fraction of relevant documents that are retrieved, and Mean Reciprocal Rank (MRR) which measures the rank of the first relevant document. A good retrieval system balances precision and recall. End-to-end RAG evaluation also includes generation metrics like faithfulness, answer relevance, and context utilization."""
    },
    {
        "id": "doc9",
        "title": "Prompt Engineering",
        "content": """Prompt engineering is the practice of designing effective inputs to language models to achieve desired outputs. For RAG systems, prompts typically include a system instruction, retrieved context, and the user question. Key techniques include chain-of-thought prompting which encourages step-by-step reasoning, few-shot prompting which provides examples, and structured output prompting which requests responses in specific formats. Context ordering affects model performance - placing most relevant context first or last tends to work better than middle placement."""
    },
    {
        "id": "doc10",
        "title": "Model Serving",
        "content": """Model serving refers to deploying trained machine learning models to serve inference requests in production. Common serving frameworks include Hugging Face Transformers, vLLM, Ollama, and TorchServe. Hugging Face Transformers provides a simple interface for loading and running models locally. vLLM is optimized for high throughput serving using PagedAttention memory management. Key serving metrics include latency (time to first token), throughput (tokens per second), and hardware utilization."""
    }
]

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  - {doc['id']}: {doc['title']}")

Loaded 10 documents
  - doc1: RAG Architecture
  - doc2: Vector Databases
  - doc3: Document Chunking
  - doc4: Embedding Models
  - doc5: LLM Hallucination
  - doc6: Agentic AI Systems
  - doc7: MLOps Pipeline
  - doc8: Retrieval Evaluation
  - doc9: Prompt Engineering
  - doc10: Model Serving


In [3]:
def chunk_documents(documents, chunk_size=512, overlap=50):
    """Split documents into chunks with overlap."""
    chunks = []
    
    for doc in documents:
        content = doc["content"]
        words = content.split()
        
        start = 0
        chunk_idx = 0
        
        while start < len(words):
            end = min(start + chunk_size, len(words))
            chunk_text = " ".join(words[start:end])
            
            chunks.append({
                "chunk_id": f"{doc['id']}_chunk{chunk_idx}",
                "doc_id": doc["id"],
                "title": doc["title"],
                "text": chunk_text,
                "chunk_idx": chunk_idx
            })
            
            if end == len(words):
                break
                
            start = end - overlap
            chunk_idx += 1
    
    return chunks

# Chunk all documents
start_time = time.time()
chunks = chunk_documents(documents, chunk_size=100, overlap=20)
chunking_time = time.time() - start_time

print(f"Created {len(chunks)} chunks in {chunking_time:.3f}s")
print(f"\nExample chunk:")
print(f"  ID: {chunks[0]['chunk_id']}")
print(f"  Title: {chunks[0]['title']}")
print(f"  Text preview: {chunks[0]['text'][:150]}...")

Created 10 chunks in 0.000s

Example chunk:
  ID: doc1_chunk0
  Title: RAG Architecture
  Text preview: Retrieval-Augmented Generation (RAG) is an AI architecture that enhances large language model outputs by retrieving relevant information from external...


In [4]:
# Load embedding model
print("Loading embedding model...")
start_time = time.time()
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embed_load_time = time.time() - start_time
print(f"Embedding model loaded in {embed_load_time:.2f}s")

# Generate embeddings for all chunks
print("\nGenerating embeddings...")
start_time = time.time()
chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = embedder.encode(chunk_texts, convert_to_numpy=True, batch_size=32)
embed_time = time.time() - start_time
print(f"Generated {len(embeddings)} embeddings of dimension {embeddings.shape[1]} in {embed_time:.3f}s")

# Build FAISS index
print("\nBuilding FAISS index...")
start_time = time.time()
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))
index_time = time.time() - start_time
print(f"FAISS index built with {index.ntotal} vectors in {index_time:.3f}s")

print(f"\n--- Indexing Summary ---")
print(f"Chunks indexed: {index.ntotal}")
print(f"Embedding dimension: {dimension}")
print(f"Embedding model load time: {embed_load_time:.2f}s")
print(f"Embedding generation time: {embed_time:.3f}s")
print(f"Index build time: {index_time:.3f}s")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded in 3.60s

Generating embeddings...
Generated 10 embeddings of dimension 384 in 0.993s

Building FAISS index...
FAISS index built with 10 vectors in 0.005s

--- Indexing Summary ---
Chunks indexed: 10
Embedding dimension: 384
Embedding model load time: 3.60s
Embedding generation time: 0.993s
Index build time: 0.005s


In [5]:
def retrieve(query, k=3):
    """Retrieve top-k relevant chunks for a query."""
    start_time = time.time()
    
    # Embed the query
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    
    # Search FAISS index
    distances, indices = index.search(query_embedding.astype(np.float32), k)
    
    retrieval_time = time.time() - start_time
    
    # Collect results
    results = []
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        results.append({
            "rank": i + 1,
            "chunk_id": chunks[idx]["chunk_id"],
            "doc_id": chunks[idx]["doc_id"],
            "title": chunks[idx]["title"],
            "text": chunks[idx]["text"],
            "distance": float(dist)
        })
    
    return results, retrieval_time

# Test retrieval
print("Testing retrieval...")
test_results, ret_time = retrieve("What is RAG and how does it work?", k=3)
print(f"Retrieval time: {ret_time*1000:.1f}ms")
print(f"\nTop 3 results:")
for r in test_results:
    print(f"  Rank {r['rank']}: [{r['title']}] (distance={r['distance']:.4f})")
    print(f"    Preview: {r['text'][:100]}...")

Testing retrieval...
Retrieval time: 23.2ms

Top 3 results:
  Rank 1: [Retrieval Evaluation] (distance=1.1472)
    Preview: Retrieval evaluation measures the quality of document retrieval in RAG systems. Key metrics include ...
  Rank 2: [RAG Architecture] (distance=1.3037)
    Preview: Retrieval-Augmented Generation (RAG) is an AI architecture that enhances large language model output...
  Rank 3: [LLM Hallucination] (distance=1.3157)
    Preview: Hallucination in large language models refers to the generation of factually incorrect or fabricated...


In [6]:
# Load Qwen2.5-7B-Instruct
print("Loading Qwen2.5-7B-Instruct...")
print("This will take 2-3 minutes...")

start_time = time.time()

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

load_time = time.time() - start_time
print(f"\nModel loaded in {load_time:.1f}s")
print(f"Model device: {next(model.parameters()).device}")

# Test quick generation
def generate_response(prompt, max_new_tokens=512):
    """Generate response from LLM."""
    start_time = time.time()
    
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only assistant response
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    
    gen_time = time.time() - start_time
    return response, gen_time

print("\nTesting generation...")
test_response, gen_time = generate_response("What is FAISS? Answer in one sentence.")
print(f"Generation time: {gen_time:.1f}s")
print(f"Response: {test_response}")
print("\nLLM ready!")

Loading Qwen2.5-7B-Instruct...
This will take 2-3 minutes...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.



Model loaded in 87.8s
Model device: cuda:0

Testing generation...
Generation time: 15.5s
Response: FAISS is a library developed by Facebook AI that implements efficient algorithms for similarity search and clustering of dense vectors.

LLM ready!


In [7]:
def rag_query(question, k=3):
    """Complete RAG pipeline: retrieve + generate grounded response."""
    
    # Step 1: Retrieve
    retrieved, retrieval_time = retrieve(question, k=k)
    
    # Step 2: Build context
    context = "\n\n".join([
        f"[{r['title']}]: {r['text']}" 
        for r in retrieved
    ])
    
    # Step 3: Build grounded prompt
    prompt = f"""Answer the question using ONLY the provided context below.
If the context does not contain enough information, say "I don't have enough information in the provided context."
Always base your answer strictly on the context.

Context:
{context}

Question: {question}

Answer:"""
    
    # Step 4: Generate
    response, generation_time = generate_response(prompt, max_new_tokens=300)
    
    # Step 5: Clean response
    if "Answer:" in response:
        response = response.split("Answer:")[-1].strip()
    
    total_time = retrieval_time + generation_time
    
    return {
        "question": question,
        "answer": response,
        "retrieved_chunks": retrieved,
        "retrieval_time_ms": retrieval_time * 1000,
        "generation_time_s": generation_time,
        "total_time_s": total_time
    }

# Test full pipeline
print("Testing full RAG pipeline...")
result = rag_query("What is RAG and how does it work?")
print(f"\nQuestion: {result['question']}")
print(f"\nAnswer: {result['answer']}")
print(f"\nLatency:")
print(f"  Retrieval: {result['retrieval_time_ms']:.1f}ms")
print(f"  Generation: {result['generation_time_s']:.1f}s")
print(f"  Total: {result['total_time_s']:.1f}s")
print(f"\nSources used:")
for r in result['retrieved_chunks']:
    print(f"  - {r['title']} (rank {r['rank']})")

Testing full RAG pipeline...

Question: What is RAG and how does it work?

Answer: RAG stands for Retrieval-Augmented Generation. It is an AI architecture designed to enhance the outputs of large language models by incorporating relevant information from external knowledge bases. The process involves two main phases: indexing and querying.

During the indexing phase, documents are broken down into smaller chunks, these chunks are then converted into vectors and stored in a vector database. This allows for efficient storage and retrieval of information based on semantic similarity.

In the querying phase, the user's query is similarly embedded and used to find the most relevant chunks from the vector database through a process known as similarity search. These selected chunks serve as contextual input for the large language model (LLM), providing the model with additional relevant information to generate more accurate and grounded responses.

By integrating this retrieved context, RAG h

In [8]:
# 10 handcrafted evaluation queries with ground truth
eval_queries = [
    {
        "question": "What is RAG and how does it work?",
        "relevant_docs": ["doc1", "doc8"]
    },
    {
        "question": "What are the different types of FAISS indexes?",
        "relevant_docs": ["doc2"]
    },
    {
        "question": "How do you choose chunk size and overlap for document chunking?",
        "relevant_docs": ["doc3"]
    },
    {
        "question": "What embedding models are available and how do they differ?",
        "relevant_docs": ["doc4"]
    },
    {
        "question": "What causes hallucination in LLMs and how can RAG help?",
        "relevant_docs": ["doc5", "doc1"]
    },
    {
        "question": "What is the ReAct pattern in agentic AI systems?",
        "relevant_docs": ["doc6"]
    },
    {
        "question": "What is MLOps and what are its key components?",
        "relevant_docs": ["doc7"]
    },
    {
        "question": "How is retrieval quality measured in RAG systems?",
        "relevant_docs": ["doc8"]
    },
    {
        "question": "What prompt engineering techniques work best for RAG?",
        "relevant_docs": ["doc9"]
    },
    {
        "question": "What are the differences between vLLM and Hugging Face Transformers for model serving?",
        "relevant_docs": ["doc10"]
    }
]

def precision_at_k(retrieved, relevant, k=3):
    top_k = [r["doc_id"] for r in retrieved[:k]]
    hits = sum(1 for doc in top_k if doc in relevant)
    return hits / k

def recall_at_k(retrieved, relevant, k=3):
    top_k = [r["doc_id"] for r in retrieved[:k]]
    hits = sum(1 for doc in top_k if doc in relevant)
    return hits / len(relevant)

def reciprocal_rank(retrieved, relevant):
    for i, r in enumerate(retrieved, 1):
        if r["doc_id"] in relevant:
            return 1.0 / i
    return 0.0

# Run evaluation
print("Running 10 evaluation queries...")
print("Note: Each query takes ~2 min for generation\n")

eval_results = []

for i, eq in enumerate(eval_queries):
    print(f"Query {i+1}/10: {eq['question'][:60]}...")
    
    result = rag_query(eq["question"], k=3)
    
    retrieved = result["retrieved_chunks"]
    relevant = set(eq["relevant_docs"])
    
    p_at_3 = precision_at_k(retrieved, relevant, k=3)
    r_at_3 = recall_at_k(retrieved, relevant, k=3)
    rr = reciprocal_rank(retrieved, relevant)
    
    eval_results.append({
        "query_id": i + 1,
        "question": eq["question"],
        "answer": result["answer"],
        "relevant_docs": eq["relevant_docs"],
        "retrieved_docs": [r["doc_id"] for r in retrieved],
        "retrieved_titles": [r["title"] for r in retrieved],
        "precision_at_3": p_at_3,
        "recall_at_3": r_at_3,
        "reciprocal_rank": rr,
        "retrieval_time_ms": result["retrieval_time_ms"],
        "generation_time_s": result["generation_time_s"],
        "total_time_s": result["total_time_s"]
    })
    
    print(f"  P@3={p_at_3:.2f} | R@3={r_at_3:.2f} | RR={rr:.2f} | Gen={result['generation_time_s']:.1f}s")

print("\nEvaluation complete!")

Running 10 evaluation queries...
Note: Each query takes ~2 min for generation

Query 1/10: What is RAG and how does it work?...
  P@3=0.67 | R@3=1.00 | RR=1.00 | Gen=148.2s
Query 2/10: What are the different types of FAISS indexes?...
  P@3=0.33 | R@3=1.00 | RR=1.00 | Gen=10.0s
Query 3/10: How do you choose chunk size and overlap for document chunki...
  P@3=0.33 | R@3=1.00 | RR=1.00 | Gen=22.8s
Query 4/10: What embedding models are available and how do they differ?...
  P@3=0.33 | R@3=1.00 | RR=1.00 | Gen=59.9s
Query 5/10: What causes hallucination in LLMs and how can RAG help?...
  P@3=0.67 | R@3=1.00 | RR=1.00 | Gen=53.9s
Query 6/10: What is the ReAct pattern in agentic AI systems?...
  P@3=0.33 | R@3=1.00 | RR=1.00 | Gen=27.0s
Query 7/10: What is MLOps and what are its key components?...
  P@3=0.33 | R@3=1.00 | RR=1.00 | Gen=50.7s
Query 8/10: How is retrieval quality measured in RAG systems?...
  P@3=0.33 | R@3=1.00 | RR=1.00 | Gen=47.3s
Query 9/10: What prompt engineering techniqu

In [9]:
# Calculate summary statistics
avg_p3 = np.mean([r["precision_at_3"] for r in eval_results])
avg_r3 = np.mean([r["recall_at_3"] for r in eval_results])
avg_mrr = np.mean([r["reciprocal_rank"] for r in eval_results])
avg_ret = np.mean([r["retrieval_time_ms"] for r in eval_results])
avg_gen = np.mean([r["generation_time_s"] for r in eval_results])
avg_total = np.mean([r["total_time_s"] for r in eval_results])

print("="*60)
print("EVALUATION SUMMARY")
print("="*60)
print(f"\nRetrieval Metrics:")
print(f"  Average Precision@3: {avg_p3:.3f}")
print(f"  Average Recall@3:    {avg_r3:.3f}")
print(f"  Average MRR:         {avg_mrr:.3f}")
print(f"\nLatency Metrics:")
print(f"  Average Retrieval:   {avg_ret:.1f}ms")
print(f"  Average Generation:  {avg_gen:.1f}s")
print(f"  Average End-to-End:  {avg_total:.1f}s")

print(f"\nPer Query Results:")
print(f"{'Q':<4} {'P@3':<6} {'R@3':<6} {'MRR':<6} {'Gen(s)':<8} {'Retrieved Docs'}")
print("-"*60)
for r in eval_results:
    docs = ", ".join(r["retrieved_titles"])[:35]
    print(f"Q{r['query_id']:<3} {r['precision_at_3']:<6.2f} {r['recall_at_3']:<6.2f} {r['reciprocal_rank']:<6.2f} {r['generation_time_s']:<8.1f} {docs}")

EVALUATION SUMMARY

Retrieval Metrics:
  Average Precision@3: 0.400
  Average Recall@3:    1.000
  Average MRR:         1.000

Latency Metrics:
  Average Retrieval:   10.7ms
  Average Generation:  46.0s
  Average End-to-End:  46.0s

Per Query Results:
Q    P@3    R@3    MRR    Gen(s)   Retrieved Docs
------------------------------------------------------------
Q1   0.67   1.00   1.00   148.2    Retrieval Evaluation, RAG Architect
Q2   0.33   1.00   1.00   10.0     Vector Databases, Retrieval Evaluat
Q3   0.33   1.00   1.00   22.8     Document Chunking, RAG Architecture
Q4   0.33   1.00   1.00   59.9     Embedding Models, Model Serving, Ve
Q5   0.67   1.00   1.00   53.9     LLM Hallucination, Retrieval Evalua
Q6   0.33   1.00   1.00   27.0     Agentic AI Systems, RAG Architectur
Q7   0.33   1.00   1.00   50.7     MLOps Pipeline, Model Serving, Agen
Q8   0.33   1.00   1.00   47.3     Retrieval Evaluation, RAG Architect
Q9   0.33   1.00   1.00   18.6     Prompt Engineering, LLM Hallucinat

In [10]:
# Save evaluation results to JSON
with open("rag_evaluation_results.json", "w") as f:
    json.dump({
        "summary": {
            "avg_precision_at_3": avg_p3,
            "avg_recall_at_3": avg_r3,
            "avg_mrr": avg_mrr,
            "avg_retrieval_ms": avg_ret,
            "avg_generation_s": avg_gen,
            "avg_total_s": avg_total
        },
        "results": eval_results
    }, f, indent=2)

print("Results saved to rag_evaluation_results.json")
print("\nRAG Pipeline notebook complete!")
print("="*60)
print("CHECKLIST:")
print("✓ Document ingestion with chunking")
print("✓ Embedding generation (all-MiniLM-L6-v2)")
print("✓ FAISS vector index")
print("✓ Retrieval with similarity search")
print("✓ Grounded generation (Qwen2.5-7B-Instruct)")
print("✓ 10 evaluation queries")
print("✓ Precision@3, Recall@3, MRR metrics")
print("✓ Latency measurements")

Results saved to rag_evaluation_results.json

RAG Pipeline notebook complete!
CHECKLIST:
✓ Document ingestion with chunking
✓ Embedding generation (all-MiniLM-L6-v2)
✓ FAISS vector index
✓ Retrieval with similarity search
✓ Grounded generation (Qwen2.5-7B-Instruct)
✓ 10 evaluation queries
✓ Precision@3, Recall@3, MRR metrics
✓ Latency measurements


In [11]:
# Save evaluation results to JSON
with open("rag_evaluation_results.json", "w") as f:
    json.dump({
        "summary": {
            "avg_precision_at_3": avg_p3,
            "avg_recall_at_3": avg_r3,
            "avg_mrr": avg_mrr,
            "avg_retrieval_ms": avg_ret,
            "avg_generation_s": avg_gen,
            "avg_total_s": avg_total
        },
        "results": eval_results
    }, f, indent=2)

print("Results saved to rag_evaluation_results.json")
print("\nRAG Pipeline notebook complete!")
print("="*60)
print("CHECKLIST:")
print("✓ Document ingestion with chunking")
print("✓ Embedding generation (all-MiniLM-L6-v2)")
print("✓ FAISS vector index")
print("✓ Retrieval with similarity search")
print("✓ Grounded generation (Qwen2.5-7B-Instruct)")
print("✓ 10 evaluation queries")
print("✓ Precision@3, Recall@3, MRR metrics")
print("✓ Latency measurements")

Results saved to rag_evaluation_results.json

RAG Pipeline notebook complete!
CHECKLIST:
✓ Document ingestion with chunking
✓ Embedding generation (all-MiniLM-L6-v2)
✓ FAISS vector index
✓ Retrieval with similarity search
✓ Grounded generation (Qwen2.5-7B-Instruct)
✓ 10 evaluation queries
✓ Precision@3, Recall@3, MRR metrics
✓ Latency measurements
